# 03 Eliashberg-Like Diagnostic And Tc Template

This notebook computes DeePTB's coupling-strength-weighted phonon spectrum from `EPCMeshData`, then demonstrates diagnostic `lambda`, `omega_log`, and McMillan / Allen-Dynes-like `Tc` estimates. This is postprocessing; it is not yet a validated full superconductivity workflow.

In [ ]:
import numpy as np

def json_default(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")


In [ ]:
from pathlib import Path
import json
import math
from dptb.postprocess.unified.eph import (
    EPCMeshData,
    Phonons,
    compute_eliashberg_spectral_function,
    compute_phonon_dos,
    compute_scattering_maps,
)

ROOT = Path.cwd()
WORK = ROOT / "work"
epc_mesh_path = WORK / "epc_mesh_data.npz"
phonons_path = WORK / "phonons_mesh.npz"
if not epc_mesh_path.exists():
    raise FileNotFoundError("Run 02_mesh_transport_mobility.ipynb first to create work/epc_mesh_data.npz")
if not phonons_path.exists():
    raise FileNotFoundError("Run 00_prepare_external_phonons.ipynb first to create work/phonons_mesh.npz")

epc_mesh = EPCMeshData.load_npz(epc_mesh_path)
phonons = Phonons.load_npz(phonons_path)
frequency_grid = np.linspace(0.01, max(50.0, float(np.nanmax(epc_mesh.frequencies)) * 1.2), 500)
sigma = 0.2
mu_star = 0.10


## Compute spectrum

In [ ]:
spectrum = compute_eliashberg_spectral_function(
    epc_mesh,
    frequency_grid=frequency_grid,
    sigma=sigma,
    broadening="gaussian",
    weighted=True,
)
(WORK / "eliashberg_like.json").write_text(json.dumps(spectrum, indent=2, default=json_default))
print(spectrum["metadata"])
print(np.asarray(spectrum["alpha2f"]).shape)


## Diagnostic Tc template

The conventional formulas require physically normalized `alpha^2F(omega)`. Current DeePTB output is a coupling-strength-weighted diagnostic spectrum, so use this cell as a template until your normalization has been validated against a reference material.

In [ ]:
omega_thz = np.asarray(spectrum["frequency_grid"], dtype=float)
alpha2f = np.asarray(spectrum["alpha2f"], dtype=float)
mask = (omega_thz > 0.0) & np.isfinite(omega_thz) & np.isfinite(alpha2f)
omega = omega_thz[mask]
a2f = alpha2f[mask]

lambda_diag = 2.0 * np.trapz(a2f / omega, omega)
if lambda_diag <= 0.0:
    raise ValueError("Diagnostic lambda is non-positive; check EPC data and normalization.")

log_omega = (2.0 / lambda_diag) * np.trapz((a2f / omega) * np.log(omega), omega)
omega_log_thz = float(np.exp(log_omega))
THZ_TO_K = 47.99243
omega_log_K = omega_log_thz * THZ_TO_K

def allen_dynes_tc(lambda_value, omega_log_kelvin, mu_star):
    denom = lambda_value - mu_star * (1.0 + 0.62 * lambda_value)
    if denom <= 0.0:
        return 0.0
    exponent = -1.04 * (1.0 + lambda_value) / denom
    return (omega_log_kelvin / 1.2) * math.exp(exponent)

tc_diag = allen_dynes_tc(lambda_diag, omega_log_K, mu_star)
print({
    "lambda_diagnostic": lambda_diag,
    "omega_log_THz": omega_log_thz,
    "omega_log_K": omega_log_K,
    "mu_star": mu_star,
    "tc_K_diagnostic": tc_diag,
})
